In [1]:
import pandas as pd
import numpy as np

In [2]:
#changing display cols and rows to see more of readout
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)

In [3]:
#importing csv file from SQL query.
#make sure csv downloaded file is named 'after_update' and in the same folder as this notebook
all_str = pd.read_csv('str_filtered_after_update.csv')
all_str.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6486 entries, 0 to 6485
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      5705 non-null   object 
 1   patient_id_model              5705 non-null   object 
 2   model_model_id                5705 non-null   object 
 3   mc_model_condition_id         5705 non-null   object 
 4   mc_model_id                   5705 non-null   object 
 5   patient_id_str                6486 non-null   object 
 6   model_condition_id            5705 non-null   object 
 7   id                            6486 non-null   object 
 8   is_reference                  6486 non-null   bool   
 9   d3s1358                       6415 non-null   object 
 10  th01                          6434 non-null   object 
 11  d21s11                        5200 non-null   object 
 12  d18s51                        6268 non-null   object 
 13  pen

In [4]:
#rename columns
all_str.rename(columns={ 
    'patient_id_model': 'model_patient_id', 
    'patient_id_str': 'str_patient_id'}, inplace=True)


In [5]:
#checking for duplicates by lab corp case number, model_condition_id, and sample reference
#shows count - lookup in gumbo by sample_reference - will show duplicates - check upload dates 
#creates new df just to examine 
upload_dupe_check = all_str.groupby(['lab_corp_case_nbr', 'model_condition_id', 'sample_reference']).size().reset_index(name='count')
upload_dupe_check.sort_values(by='count', ascending=False).head(299)

,lab_corp_case_nbr,model_condition_id,sample_reference,count
2598,CX4016248,MC-001465-V7hs,CHLA136-enCas12a,2
1429,CX4012569,MC-002495-ZD9E,IOWA1T,2
2055,CX4014681,MC-003201-2p67,C42,2
3077,CX4017930,MC-002433-iccX,CCLFPEDS0019T-enCas12a-HumagneCD-RepB-P6(D21),2
2012,CX4014597,MC-000643-Lgxt,HDQP1,2
1555,CX4013150,MC-003072-zgEn,CHLA05ATRT,2
2062,CX4014682,MC-003118-NF4c,HMEC1,2
1808,CX4013811,MC-003016-CoPk,TELOHAEC,2
2418,CX4015873,MC-001212-ItHM,TTC642-311Cas9(5/30/2017),2
1567,CX4013151,MC-003061-5LP5,NCIH774,2


In [6]:
all_str.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6486 entries, 0 to 6485
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      5705 non-null   object 
 1   model_patient_id              5705 non-null   object 
 2   model_model_id                5705 non-null   object 
 3   mc_model_condition_id         5705 non-null   object 
 4   mc_model_id                   5705 non-null   object 
 5   str_patient_id                6486 non-null   object 
 6   model_condition_id            5705 non-null   object 
 7   id                            6486 non-null   object 
 8   is_reference                  6486 non-null   bool   
 9   d3s1358                       6415 non-null   object 
 10  th01                          6434 non-null   object 
 11  d21s11                        5200 non-null   object 
 12  d18s51                        6268 non-null   object 
 13  pen

In [7]:
#reformatting upload_date to remove seconds 
all_str_single_upload = all_str
all_str_single_upload['created_at'] = pd.to_datetime(all_str_single_upload['created_at']).dt.strftime('%Y-%m-%d %H:%M')

In [8]:
#showing upload dates - year, month, day, hour and minute 
# showing upload counts
all_str_single_upload['created_at'] = pd.to_datetime(all_str_single_upload['created_at'])
all_str_single_upload_sort = all_str_single_upload.sort_values(by='created_at')
counts = all_str_single_upload_sort['created_at'].value_counts().sort_index()
counts

created_at
2024-04-09 16:00:00    3742
2024-04-10 17:49:00       1
2024-04-10 17:59:00       4
2024-04-10 18:00:00      42
2024-04-10 18:01:00      37
2024-04-10 18:02:00      42
2024-04-10 18:03:00      38
2024-04-10 18:04:00      22
2024-04-10 18:05:00      12
2024-04-10 18:06:00       3
2024-04-10 18:07:00       5
2024-04-10 18:08:00       3
2024-04-10 18:11:00       2
2024-04-10 18:12:00       2
2024-04-10 18:13:00       1
2024-04-10 18:14:00       1
2024-04-10 18:20:00       1
2024-04-10 18:33:00       2
2024-04-10 18:38:00       1
2024-04-10 19:12:00      38
2024-04-10 19:13:00      10
2024-04-10 19:15:00       2
2024-04-10 19:34:00      20
2024-04-10 19:35:00       1
2024-04-10 19:48:00       3
2024-04-10 20:12:00      18
2024-04-23 18:08:00       5
2024-04-23 18:09:00       7
2024-04-23 18:10:00       1
2024-04-23 18:22:00       3
2024-04-24 16:40:00       1
2024-04-24 16:41:00       1
2024-04-24 16:58:00       1
2024-04-29 14:57:00       1
2024-05-01 16:38:00      11
2024-05-0

In [9]:
# stripping created_at column of hour and minute - just the year-month-day
all_str_single_upload = all_str
all_str_single_upload['created_at'] = pd.to_datetime(all_str_single_upload['created_at']).dt.date


In [10]:
#using date filter of 8-13-2024 - can edit this date
filter_date = pd.to_datetime('2024-08-13').date()

In [11]:
#separating df into two separate dataframes - one containing filter_date and one without filter_dat
str_containing_filter_date = all_str_single_upload[all_str_single_upload['created_at'] == filter_date]
remaining_df = all_str_single_upload[all_str_single_upload['created_at'] != filter_date]

In [12]:
#check
remaining_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4406 entries, 0 to 4405
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      3625 non-null   object 
 1   model_patient_id              3625 non-null   object 
 2   model_model_id                3625 non-null   object 
 3   mc_model_condition_id         3625 non-null   object 
 4   mc_model_id                   3625 non-null   object 
 5   str_patient_id                4406 non-null   object 
 6   model_condition_id            3625 non-null   object 
 7   id                            4406 non-null   object 
 8   is_reference                  4406 non-null   bool   
 9   d3s1358                       4338 non-null   object 
 10  th01                          4357 non-null   object 
 11  d21s11                        3132 non-null   object 
 12  d18s51                        4191 non-null   object 
 13  penta_e 

In [13]:
#check
str_containing_filter_date.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2080 entries, 4406 to 6485
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      2080 non-null   object 
 1   model_patient_id              2080 non-null   object 
 2   model_model_id                2080 non-null   object 
 3   mc_model_condition_id         2080 non-null   object 
 4   mc_model_id                   2080 non-null   object 
 5   str_patient_id                2080 non-null   object 
 6   model_condition_id            2080 non-null   object 
 7   id                            2080 non-null   object 
 8   is_reference                  2080 non-null   bool   
 9   d3s1358                       2077 non-null   object 
 10  th01                          2077 non-null   object 
 11  d21s11                        2068 non-null   object 
 12  d18s51                        2077 non-null   object 
 13  penta

In [14]:
#dropping duplicates in the str_containing_filter_date based on 'sample_reference' and 'lab_corp_case_nbr'
str_containing_filter_date = str_containing_filter_date.drop_duplicates(subset=['sample_reference', 'lab_corp_case_nbr'], keep='first')
str_containing_filter_date.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2080 entries, 4406 to 6485
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      2080 non-null   object 
 1   model_patient_id              2080 non-null   object 
 2   model_model_id                2080 non-null   object 
 3   mc_model_condition_id         2080 non-null   object 
 4   mc_model_id                   2080 non-null   object 
 5   str_patient_id                2080 non-null   object 
 6   model_condition_id            2080 non-null   object 
 7   id                            2080 non-null   object 
 8   is_reference                  2080 non-null   bool   
 9   d3s1358                       2077 non-null   object 
 10  th01                          2077 non-null   object 
 11  d21s11                        2068 non-null   object 
 12  d18s51                        2077 non-null   object 
 13  penta

In [15]:
#combining the str_containing_filter_date that no longer has dupes wiht the remaining_df
all_str_single_upload = pd.concat([str_containing_filter_date, remaining_df]).drop_duplicates()
all_str_single_upload.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6486 entries, 4406 to 4405
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      5705 non-null   object 
 1   model_patient_id              5705 non-null   object 
 2   model_model_id                5705 non-null   object 
 3   mc_model_condition_id         5705 non-null   object 
 4   mc_model_id                   5705 non-null   object 
 5   str_patient_id                6486 non-null   object 
 6   model_condition_id            5705 non-null   object 
 7   id                            6486 non-null   object 
 8   is_reference                  6486 non-null   bool   
 9   d3s1358                       6415 non-null   object 
 10  th01                          6434 non-null   object 
 11  d21s11                        5200 non-null   object 
 12  d18s51                        6268 non-null   object 
 13  penta

In [16]:
#resetting index
all_str_single_upload = all_str_single_upload.reset_index(drop=True)
all_str_single_upload.head(10)

,model_id,model_patient_id,model_model_id,mc_model_condition_id,mc_model_id,str_patient_id,model_condition_id,id,is_reference,d3s1358,th01,d21s11,d18s51,penta_e,d5s818,d13s317,d7s820,d16s539,csf1po,penta_d,vwa,d8s1179,tpox,fga,pellet_creation_date,percentage_match_to_parental,lab_corp_spec_nbr,lab_corp_case_nbr,comments,amelogenin,model_condition_id.1,mouse,mycoplasma,patient_id,source,source_group,pellet_submitted_date,sample_reference,created_at,updated_at
0,ACH-000296,PT-GezIUF,ACH-000296,MC-000296-bgzW,ACH-000296,PT-GezIUF,MC-000296-bgzW,STR-WWzBpU,False,"16, 17",9.3,"28, 30",16,5,11,12,10,11,12,"9, 12",16,"13, 16",11,23.2,2024-03-11,NaN,NaN,CX4018877,NaN,X,MC-000296-bgzW,NaN,NaN,PT-GezIUF,Internal,GPP,NaN,OUMS23Cas9VY,2024-08-13,2024-08-13 15:46:14.431081+00:00
1,ACH-002499,PT-j6UudO,ACH-002499,MC-002499-dWxK,ACH-002499,PT-j6UudO,MC-002499-dWxK,STR-U7lF8I,False,18,9.3,30.2,20,12,13,11,10,12,12,12,"15, 17","11, 13","10, 13",23,NaN,NaN,NaN,CX4012569,NaN,X,MC-002499-dWxK,NaN,NaN,PT-j6UudO,Internal,GPP,NaN,UACC3199-311cas9,2024-08-13,2024-08-13 14:32:29.862597+00:00
2,ACH-002499,PT-j6UudO,ACH-002499,MC-002499-dWxK,ACH-002499,PT-j6UudO,MC-002499-dWxK,STR-QADpM0,False,18,9.3,30.2,20,12,13,11,10,12,12,12,"15, 17","11, 13","10, 13",23,2022-08-31,NaN,NaN,CX4014596,NaN,X,MC-002499-dWxK,NaN,NaN,PT-j6UudO,Internal,GPP,NaN,UACC3199-311Cas9-RepA-P4,2024-08-13,2024-08-13 14:32:29.862597+00:00
3,ACH-002051,PT-K31XId,ACH-002051,MC-002051-2Ddi,ACH-002051,PT-K31XId,MC-002051-2Ddi,STR-lKKdW7,False,17,9,30,17,14,11,9,11,9,12,"9, 13",17,"10, 11",11,19,NaN,NaN,NaN,CX4011714,NaN,X,MC-002051-2Ddi,NaN,NaN,PT-K31XId,Internal,GPP,NaN,LU134A-311cas9,2024-08-13,2024-08-13 14:32:29.862597+00:00
4,ACH-003170,PT-FJLnco,ACH-003170,MC-003170-0VvA,ACH-003170,PT-FJLnco,MC-003170-0VvA,STR-b5TCvd,False,"15, 16",9.3,"26.2, 31",13,"13, 18",11,12,"10, 11","11, 13","11, 12","9, 15","15, 16",14,8,"19, 21",2023-12-09,NaN,NaN,CX4018283,NaN,X,MC-003170-0VvA,NaN,NaN,PT-FJLnco,Internal,GPP,NaN,NRHLS1-enCas12a,2024-08-13,2024-08-13 15:45:03.795747+00:00
5,ACH-000085,PT-FnO2p8,ACH-000085,MC-000085-DjQm,ACH-000085,PT-FnO2p8,MC-000085-DjQm,STR-1EcCBm,False,14,9.3,32.2,16,"11, 12",11,"10, 13",12,"11, 12",10,10,"18, 19, 21",16,"11, 12","24, 25",2023-04-21,NaN,NaN,CX4016249,NaN,X,MC-000085-DjQm,NaN,NaN,PT-FnO2p8,Internal,GPP,NaN,T3M4-Cas9VY,2024-08-13,2024-08-13 15:45:03.795747+00:00
6,ACH-000085,PT-FnO2p8,ACH-000085,MC-000085-DjQm,ACH-000085,PT-FnO2p8,MC-000085-DjQm,STR-0tHUMB,False,14,9.3,32.2,16,"11, 12",11,"10, 13",12,"11, 12",10,10,"18, 21",16,"11, 12","24, 25",2023-09-07,NaN,NaN,CX4017466,NaN,X,MC-000085-DjQm,NaN,NaN,PT-FnO2p8,Internal,GPP,NaN,T3M4-Cas9VY-ParalogV2-RepA-P6(D22),2024-08-13,2024-08-13 15:45:03.795747+00:00
7,ACH-002708,PT-03AcoW,ACH-002708,MC-002708-tPxj,ACH-002708,PT-03AcoW,MC-002708-tPxj,STR-gXdz7j,False,"15, 16","8, 9",29,"14, 17","12, 16",12,"8, 11","10, 12","12, 13","10, 13","10, 12","14, 17","10, 14",8,20,NaN,NaN,NaN,CX4011254,NaN,X,MC-002708-tPxj,NaN,NaN,PT-03AcoW,Internal,GPP,NaN,DL221-311cas9-RepA-p6,2024-08-13,2024-08-13 15:00:15.893191+00:00
8,ACH-002708,PT-03AcoW,ACH-002708,MC-002708-tPxj,ACH-002708,PT-03AcoW,MC-002708-tPxj,STR-rZZq4u,False,"15, 16","8, 9",29,"14, 17","12, 16",12,"8, 11","10, 12","12, 13","10, 13","10, 12","14, 17","10, 14",8,20,NaN,NaN,NaN,CX4011254,NaN,X,MC-002708-tPxj,NaN,NaN,PT-03AcoW,Internal,GPP,NaN,DL221-311cas9-RepB-p6,2024-08-13,2024-08-13 15:00:15.893191+00:00
9,ACH-002708,PT-03AcoW,ACH-002708,MC-002708-tPxj,ACH-002708,PT-03AcoW,MC-002708-tPxj,STR-42bCys,False,"15, 16","8, 9",29,"14, 17","12, 16",12,"8, 11","10, 12","12, 13","10, 13","10, 12","14, 17","10, 14",8,20,NaN,NaN,NaN,CX4010487,NaN,X,MC-002708-tPxj,NaN,NaN,PT-03AcoW,Internal,GPP,NaN,DL221-311cas9,2024-08-13,2024-08-13 15:00:15.893191+00:00


In [17]:
all_str_single_upload.columns

Index(['model_id', 'model_patient_id', 'model_model_id',
       'mc_model_condition_id', 'mc_model_id', 'str_patient_id',
       'model_condition_id', 'id', 'is_reference', 'd3s1358', 'th01', 'd21s11',
       'd18s51', 'penta_e', 'd5s818', 'd13s317', 'd7s820', 'd16s539', 'csf1po',
       'penta_d', 'vwa', 'd8s1179', 'tpox', 'fga', 'pellet_creation_date',
       'percentage_match_to_parental', 'lab_corp_spec_nbr',
       'lab_corp_case_nbr', 'comments', 'amelogenin', 'model_condition_id.1',
       'mouse', 'mycoplasma', 'patient_id', 'source', 'source_group',
       'pellet_submitted_date', 'sample_reference', 'created_at',
       'updated_at'],
      dtype='object')

In [18]:
all_str_single_upload.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6486 entries, 0 to 6485
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      5705 non-null   object 
 1   model_patient_id              5705 non-null   object 
 2   model_model_id                5705 non-null   object 
 3   mc_model_condition_id         5705 non-null   object 
 4   mc_model_id                   5705 non-null   object 
 5   str_patient_id                6486 non-null   object 
 6   model_condition_id            5705 non-null   object 
 7   id                            6486 non-null   object 
 8   is_reference                  6486 non-null   bool   
 9   d3s1358                       6415 non-null   object 
 10  th01                          6434 non-null   object 
 11  d21s11                        5200 non-null   object 
 12  d18s51                        6268 non-null   object 
 13  pen

In [19]:
#checking for patient ids that dont match - including rows where one column has a na
mask = ((all_str_single_upload['model_patient_id'] != all_str_single_upload['str_patient_id']) |
    (all_str_single_upload['model_patient_id'].isna() != all_str_single_upload['str_patient_id'].isna()))
patient_id_discrepancies_including_na = all_str_single_upload[mask]
patient_id_discrepancies_including_na.head(15)

,model_id,model_patient_id,model_model_id,mc_model_condition_id,mc_model_id,str_patient_id,model_condition_id,id,is_reference,d3s1358,th01,d21s11,d18s51,penta_e,d5s818,d13s317,d7s820,d16s539,csf1po,penta_d,vwa,d8s1179,tpox,fga,pellet_creation_date,percentage_match_to_parental,lab_corp_spec_nbr,lab_corp_case_nbr,comments,amelogenin,model_condition_id.1,mouse,mycoplasma,patient_id,source,source_group,pellet_submitted_date,sample_reference,created_at,updated_at
2225,NaN,NaN,NaN,NaN,NaN,PT-vEZ2Aw,NaN,STR-wfpoXW,False,"15, 16","6, 9.3","27, 30","14, 17","7, 11",12,"10, 11","9, 10","12, 13",10,"11, 15","15, 16","11, 14",11,"21, 22",NaN,NaN,77I90570,CX4003713,unique; Unique cell line - no reference profil...,"X, Y",NaN,NaN,NaN,PT-vEZ2Aw,Internal,PRISM,NaN,C8166_LUA-4915,2024-04-09,2024-04-09 16:00:46.668811+00:00
2256,NaN,NaN,NaN,NaN,NaN,PT-xfVhgW,NaN,STR-0aaUxd,False,"14, 17","6, 9",30,"17, 19","5, 10","9, 13","9, 13","9, 11",13,"10, 11","11, 14",18,"12, 14","8, 11",23,NaN,NaN,82192670,CX4004605,matches reference; STR matches reference profi...,X,NaN,Not Detected,NaN,PT-xfVhgW,Internal,PRISM,NaN,TE 617.T_LUA-4867,2024-04-09,2024-04-09 16:00:46.668811+00:00
2277,NaN,NaN,NaN,NaN,NaN,PT-pktZZ0,NaN,STR-0C5Man,True,18,9,31.2,14,14,12,11,"8, 10",11,12,12,"14, 15","11, 13","8, 12",19,2022-07-11,NaN,27F91120,CX4014235,NaN,"X, Y",NaN,Not Detected,NaN,PT-pktZZ0,Internal,Mnemosyne,2022-07-14,ACH-002815-Pre-O (p6),2024-04-09,2024-04-09 16:00:46.668811+00:00
2290,NaN,NaN,NaN,NaN,NaN,PT-Mi0cKz,NaN,STR-0DiPWv,False,16,6,29,"16, 18","7, 10","10, 12",12,"10, 11",9,"10, 12","11, 15","17, 19",14,"8, 11","17, 22",NaN,NaN,7A590840,CX4004072,matches reference; STR matches reference profile,X,NaN,NaN,NaN,PT-Mi0cKz,Internal,PRISM,NaN,T24_LUA-3463,2024-04-09,2024-04-09 16:00:46.668811+00:00
2355,NaN,NaN,NaN,NaN,NaN,PT-j60QIc,NaN,STR-HoomuX,False,15,9.3,"28, 29, 30","15, 19","13, 15","12, 13","10, 13","8, 10","11, 13",10,"11, 12",18,"13, 15",12,"20, 21",NaN,NaN,84C92190,CX4004964,matches reference; STR matches reference profi...,"X, Y",NaN,Not Detected,NaN,PT-j60QIc,Internal,PRISM,NaN,WM-88_LUA-3105,2024-04-09,2024-04-09 16:00:46.668811+00:00
2387,NaN,NaN,NaN,NaN,NaN,PT-OfCVRv,NaN,STR-kM6u1h,True,15,9.3,"29, 30","12, 14","7, 14",11,11,"9, 11","11, 12",10,"12, 13","18, 19","11, 13",11,"19, 22",NaN,NaN,7AP91410,CX4004205,matches reference; Matches reference profile a...,X,NaN,Not Detected,NaN,PT-OfCVRv,Internal,DMX,NaN,184B5; From Legacy pellet MC-001827-qPLL,2024-04-09,2024-06-10 19:23:55.122242+00:00
2423,NaN,NaN,NaN,NaN,NaN,PT-kI5CjI,NaN,STR-hvDm3j,False,16,9,31.2,12,"15, 17",12,"11, 12","8, 12","11, 13","10, 12",13,17,"12, 13","8, 11",18,NaN,NaN,87691080,CX4005370,matches reference; STR matches reference profile,"X, Y",NaN,Not Detected,NaN,PT-kI5CjI,Internal,PRISM,NaN,A427_LUA-3196,2024-04-09,2024-04-09 16:00:46.668811+00:00
2446,NaN,NaN,NaN,NaN,NaN,PT-4qxYZb,NaN,STR-0vRjam,False,15,7,27,15,"7, 13",12,11,"9, 12",11,12,"2.2, 8",15,"12, 13","8, 11","19, 22",NaN,NaN,7A591310,CX4004074,matches reference; STR matches reference profile,X,NaN,NaN,NaN,PT-4qxYZb,Internal,PRISM,NaN,NCI-H2172_LUA-3983,2024-04-09,2024-04-09 16:00:46.668811+00:00
2492,NaN,NaN,NaN,NaN,NaN,PT-5LVIPg,NaN,STR-jvpSct,False,18,9.3,30,10,12,11,11,"8, 9","11, 12",11,8,18,12,11,20,NaN,NaN,7A591210,CX4004073,matches reference; STR matches reference profile,X,NaN,NaN,NaN,PT-5LVIPg,Internal,PRISM,NaN,NCI-H1650_LUA-3154,2024-04-09,2024-04-09 16:00:46.668811+00:00
2507,NaN,NaN,NaN,NaN,NaN,PT-T9nCW5,NaN,STR-17qBvb,False,16,9,31,"12, 16","5, 14",11,11,10,13,12,13,17,"10, 14",8,22,NaN,NaN,84C91470,CX4004961,matches reference; STR matches reference profi...,X,NaN,Not Detected,NaN,PT-T9nCW5,Internal,PRISM,NaN,Calu-6_LUA-3040,2024-04-09,2024-04-09 16:00:46.668811+00:00


In [20]:
#saving as csv file
patient_id_discrepancies_including_na = patient_id_discrepancies_including_na.reset_index(drop=True)
patient_id_discrepancies_including_na.to_csv('patient_id_discrepancies_including_na.csv',index=False)
patient_id_discrepancies_including_na.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 789 entries, 0 to 788
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      8 non-null      object 
 1   model_patient_id              8 non-null      object 
 2   model_model_id                8 non-null      object 
 3   mc_model_condition_id         8 non-null      object 
 4   mc_model_id                   8 non-null      object 
 5   str_patient_id                789 non-null    object 
 6   model_condition_id            8 non-null      object 
 7   id                            789 non-null    object 
 8   is_reference                  789 non-null    bool   
 9   d3s1358                       774 non-null    object 
 10  th01                          788 non-null    object 
 11  d21s11                        779 non-null    object 
 12  d18s51                        768 non-null    object 
 13  penta

In [21]:
#checking for patient ids that dont match - patient id are different (not including na)
mask = ( (all_str_single_upload['model_patient_id'] != all_str_single_upload['str_patient_id']) &
      (all_str_single_upload['model_patient_id'].notna() & all_str_single_upload['str_patient_id'].notna()))
patient_id_discrepancies = all_str_single_upload[mask]
patient_id_discrepancies

,model_id,model_patient_id,model_model_id,mc_model_condition_id,mc_model_id,str_patient_id,model_condition_id,id,is_reference,d3s1358,th01,d21s11,d18s51,penta_e,d5s818,d13s317,d7s820,d16s539,csf1po,penta_d,vwa,d8s1179,tpox,fga,pellet_creation_date,percentage_match_to_parental,lab_corp_spec_nbr,lab_corp_case_nbr,comments,amelogenin,model_condition_id.1,mouse,mycoplasma,patient_id,source,source_group,pellet_submitted_date,sample_reference,created_at,updated_at
2554,ACH-003298,PT-a93e6k,ACH-003298,MC-003298-iops,ACH-003298,PT-DzY90c,MC-003298-iops,STR-wDl1PX,False,"17, 18","7, 9.3",NaN,"12, 17","7.1, 8.1","9, 11","12, 13","11, 12","11, 13","11, 12",NaN,NaN,"12, 14",11,"21, 25",NaN,NaN,NaN,NaN,NaN,NaN,MC-003298-iops,NaN,NaN,PT-DzY90c,Internal,WGS Inferred,NaN,NaN,2024-04-10,2024-04-10 18:01:27.529945+00:00
3843,ACH-003298,PT-a93e6k,ACH-003298,MC-003298-iops,ACH-003298,PT-DzY90c,MC-003298-iops,STR-KcLKO2,False,"17, 18","9.3, 7",NaN,"12, 17","7.1, 8.1","11, 9","12, 13","12, 11","11, 13","12, 11",NaN,NaN,"12, 14",11,"25, 21",NaN,NaN,NaN,NaN,NaN,NaN,MC-003298-iops,NaN,NaN,PT-DzY90c,Internal,WGS Inferred,NaN,NaN,2024-04-09,2024-04-09 16:00:46.668811+00:00
4387,ACH-000956,PT-FFSOW0,ACH-000956,MC-000956-WA5g,ACH-000956,PT-rqTCjv,MC-000956-WA5g,STR-ynUjMG,False,15,"6, 9.3",31.3,"13, 14",5.1,NaN,"9, 12","11, 9",12,10,"11, 8",NaN,"13, 14",8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MC-000956-WA5g,NaN,NaN,PT-rqTCjv,Internal,WGS Inferred,NaN,NaN,2024-04-09,2024-04-09 16:00:46.668811+00:00
4436,ACH-002851,PT-QKOh5t,ACH-002851,MC-002851-Vej0,ACH-002851,PT-wKJvCj,MC-002851-Vej0,STR-R063A0,False,"15, 17",6,NaN,14,"12.1, 7.1","10, 11","9, 14","9, 12","11, 12","10, 11",12,NaN,"13, 14","9, 8","20, 21",NaN,NaN,NaN,NaN,NaN,NaN,MC-002851-Vej0,NaN,NaN,PT-wKJvCj,Internal,WGS Inferred,NaN,NaN,2024-04-09,2024-04-09 16:00:46.668811+00:00
4790,ACH-002851,PT-QKOh5t,ACH-002851,MC-002851-Vej0,ACH-002851,PT-wKJvCj,MC-002851-Vej0,STR-dWuqsd,False,"15, 17",6,NaN,14,"7.1, 12.1","10, 11","9, 14","12, 9","11, 12","10, 11","11, 12",NaN,"13, 14","8, 9","21, 20",NaN,NaN,NaN,NaN,NaN,NaN,MC-002851-Vej0,NaN,NaN,PT-wKJvCj,Internal,WGS Inferred,NaN,NaN,2024-04-09,2024-04-09 16:00:46.668811+00:00
5702,ACH-003370,PT-QKBqJJ,ACH-003370,MC-003370-hUep,ACH-003370,PT-l7O5zs,MC-003370-hUep,STR-neD4je,False,"15, 17","6, 9.3",NaN,"15, 20",7.1,"12, 13","9, 12",10,"9, 13","11, 12",10,NaN,"10, 13",NaN,"20, 23",NaN,NaN,NaN,NaN,NaN,NaN,MC-003370-hUep,NaN,NaN,PT-l7O5zs,Internal,WGS Inferred,NaN,NaN,2024-05-01,2024-05-01 16:39:43.608810+00:00
6067,ACH-002923,PT-22FXN6,ACH-002923,MC-002923-TlcE,ACH-002923,PT-5fEWVP,MC-002923-TlcE,STR-tByA3I,False,"16, 16.2",9.3,31.3,14,"11.1, 12.1",12,"11, 12","10, 11","9, 10",12,"8, 12",NaN,"13, 14",NaN,19,NaN,NaN,NaN,NaN,NaN,NaN,MC-002923-TlcE,NaN,NaN,PT-5fEWVP,Internal,WGS Inferred,NaN,NaN,2024-05-22,2024-05-22 16:39:41.348395+00:00
6309,ACH-000000,PT-Q3O6VZ,ACH-000000,MC-000000-TEST,ACH-000000,PT-01q6LP,MC-000000-TEST,STR-CJTFQN,False,TEST1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MC-000000-TEST,NaN,NaN,PT-01q6LP,Internal,NaN,NaN,NaN,2024-07-22,2024-07-22 20:41:46.424788+00:00


In [22]:
#saving as csv file
patient_id_discrepancies = patient_id_discrepancies.reset_index(drop=True)
patient_id_discrepancies.to_csv('patient_id_discrepancies.csv',index=False)
patient_id_discrepancies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      8 non-null      object 
 1   model_patient_id              8 non-null      object 
 2   model_model_id                8 non-null      object 
 3   mc_model_condition_id         8 non-null      object 
 4   mc_model_id                   8 non-null      object 
 5   str_patient_id                8 non-null      object 
 6   model_condition_id            8 non-null      object 
 7   id                            8 non-null      object 
 8   is_reference                  8 non-null      bool   
 9   d3s1358                       8 non-null      object 
 10  th01                          7 non-null      object 
 11  d21s11                        2 non-null      object 
 12  d18s51                        7 non-null      object 
 13  penta_e  

In [23]:
#checking for model ids that dont match
mask_model = ( (all_str_single_upload['model_model_id'] != all_str_single_upload['mc_model_id']) &
      (all_str_single_upload['model_model_id'].notna() & all_str_single_upload['mc_model_id'].notna()))
model_id_discrepancies = all_str_single_upload[mask_model]
model_id_discrepancies

,model_id,model_patient_id,model_model_id,mc_model_condition_id,mc_model_id,str_patient_id,model_condition_id,id,is_reference,d3s1358,th01,d21s11,d18s51,penta_e,d5s818,d13s317,d7s820,d16s539,csf1po,penta_d,vwa,d8s1179,tpox,fga,pellet_creation_date,percentage_match_to_parental,lab_corp_spec_nbr,lab_corp_case_nbr,comments,amelogenin,model_condition_id.1,mouse,mycoplasma,patient_id,source,source_group,pellet_submitted_date,sample_reference,created_at,updated_at


In [24]:
#saving as csv file
model_id_discrepancies = model_id_discrepancies.reset_index(drop=True)
model_id_discrepancies.to_csv('model_id_discrepancies.csv',index=False)
model_id_discrepancies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      0 non-null      object 
 1   model_patient_id              0 non-null      object 
 2   model_model_id                0 non-null      object 
 3   mc_model_condition_id         0 non-null      object 
 4   mc_model_id                   0 non-null      object 
 5   str_patient_id                0 non-null      object 
 6   model_condition_id            0 non-null      object 
 7   id                            0 non-null      object 
 8   is_reference                  0 non-null      bool   
 9   d3s1358                       0 non-null      object 
 10  th01                          0 non-null      object 
 11  d21s11                        0 non-null      object 
 12  d18s51                        0 non-null      object 
 13  penta_e          

In [99]:
all_str_single_upload.to_csv('all_str_single_upload.csv', index=False)
all_str_single_upload.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6486 entries, 0 to 6485
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      5705 non-null   object 
 1   model_patient_id              5705 non-null   object 
 2   model_model_id                5705 non-null   object 
 3   mc_model_condition_id         5705 non-null   object 
 4   mc_model_id                   5705 non-null   object 
 5   str_patient_id                6486 non-null   object 
 6   model_condition_id            5705 non-null   object 
 7   id                            6486 non-null   object 
 8   is_reference                  6486 non-null   bool   
 9   d3s1358                       6415 non-null   object 
 10  th01                          6434 non-null   object 
 11  d21s11                        5200 non-null   object 
 12  d18s51                        6268 non-null   object 
 13  pen

In [26]:
#checking for model condition ids that dont match
mask_model_condition = ( (all_str_single_upload['mc_model_condition_id'] != all_str_single_upload['model_condition_id']) &
      (all_str_single_upload['mc_model_condition_id'].notna() & all_str_single_upload['model_condition_id'].notna()))
model_condition_id_discrepancies = all_str_single_upload[mask_model_condition]
model_condition_id_discrepancies

,model_id,model_patient_id,model_model_id,mc_model_condition_id,mc_model_id,str_patient_id,model_condition_id,id,is_reference,d3s1358,th01,d21s11,d18s51,penta_e,d5s818,d13s317,d7s820,d16s539,csf1po,penta_d,vwa,d8s1179,tpox,fga,pellet_creation_date,percentage_match_to_parental,lab_corp_spec_nbr,lab_corp_case_nbr,comments,amelogenin,model_condition_id.1,mouse,mycoplasma,patient_id,source,source_group,pellet_submitted_date,sample_reference,created_at,updated_at


In [27]:
#saving as csv file
model_condition_id_discrepancies = model_condition_id_discrepancies.reset_index(drop=True)
model_condition_id_discrepancies.to_csv('model_condition_id_discrepancies.csv',index=False)
model_condition_id_discrepancies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      0 non-null      object 
 1   model_patient_id              0 non-null      object 
 2   model_model_id                0 non-null      object 
 3   mc_model_condition_id         0 non-null      object 
 4   mc_model_id                   0 non-null      object 
 5   str_patient_id                0 non-null      object 
 6   model_condition_id            0 non-null      object 
 7   id                            0 non-null      object 
 8   is_reference                  0 non-null      bool   
 9   d3s1358                       0 non-null      object 
 10  th01                          0 non-null      object 
 11  d21s11                        0 non-null      object 
 12  d18s51                        0 non-null      object 
 13  penta_e          

In [28]:
#missing model conditions
missing_model_cond_id = all_str_single_upload[all_str_single_upload['mc_model_condition_id'].isna()]
missing_model_cond_id = missing_model_cond_id.reset_index(drop=True)
missing_model_cond_id.to_csv('missing_model_cond_id.csv')
missing_model_cond_id.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 781 entries, 0 to 780
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      0 non-null      object 
 1   model_patient_id              0 non-null      object 
 2   model_model_id                0 non-null      object 
 3   mc_model_condition_id         0 non-null      object 
 4   mc_model_id                   0 non-null      object 
 5   str_patient_id                781 non-null    object 
 6   model_condition_id            0 non-null      object 
 7   id                            781 non-null    object 
 8   is_reference                  781 non-null    bool   
 9   d3s1358                       766 non-null    object 
 10  th01                          781 non-null    object 
 11  d21s11                        777 non-null    object 
 12  d18s51                        761 non-null    object 
 13  penta

In [29]:
#STRs with percentage match <0.75
df_under75 = all_str_single_upload[all_str_single_upload['percentage_match_to_parental'] < 0.75]
df_under75.info()

<class 'pandas.core.frame.DataFrame'>
Index: 245 entries, 574 to 6420
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      231 non-null    object 
 1   model_patient_id              231 non-null    object 
 2   model_model_id                231 non-null    object 
 3   mc_model_condition_id         231 non-null    object 
 4   mc_model_id                   231 non-null    object 
 5   str_patient_id                245 non-null    object 
 6   model_condition_id            231 non-null    object 
 7   id                            245 non-null    object 
 8   is_reference                  245 non-null    bool   
 9   d3s1358                       228 non-null    object 
 10  th01                          236 non-null    object 
 11  d21s11                        146 non-null    object 
 12  d18s51                        206 non-null    object 
 13  penta_e

In [30]:
df_under75

,model_id,model_patient_id,model_model_id,mc_model_condition_id,mc_model_id,str_patient_id,model_condition_id,id,is_reference,d3s1358,th01,d21s11,d18s51,penta_e,d5s818,d13s317,d7s820,d16s539,csf1po,penta_d,vwa,d8s1179,tpox,fga,pellet_creation_date,percentage_match_to_parental,lab_corp_spec_nbr,lab_corp_case_nbr,comments,amelogenin,model_condition_id.1,mouse,mycoplasma,patient_id,source,source_group,pellet_submitted_date,sample_reference,created_at,updated_at
574,ACH-000024,PT-codP7n,ACH-000024,MC-000024-kxGf,ACH-000024,PT-codP7n,MC-000024-kxGf,STR-KfHBSk,False,"15, 18","6, 7","30, 33.2, 34.2","13, 14","12, 21","1, 12","1, 10","1, 11","9, 13","12, 13","1, 8","14, 17","10, 13","1, 7","20, 21",2023-03-31,0.6792,NaN,CX4016035,NaN,X,MC-000024-kxGf,NaN,NaN,PT-codP7n,Internal,GPP,NaN,OPM2-311cas9(6/8/2018),2024-08-13,2024-08-13 14:32:29.535919+00:00
673,ACH-000243,PT-dLpuzD,ACH-000243,MC-000243-pHKG,ACH-000243,PT-dLpuzD,MC-000243-pHKG,STR-cUClKj,False,"1, 15","1, 8","29, 31.2, 32.2","1, 15","1, 6","12, 13","1, 7","10, 13","8, 11","1, 12","9, 13","16, 18","10, 11","1, 9","1, 19",2023-03-27,0.5200,NaN,CX4016034,NaN,X,MC-000243-pHKG,NaN,NaN,PT-dLpuzD,Internal,GPP,NaN,DANG-Cas9VY,2024-08-13,2024-08-13 14:32:29.586872+00:00
888,ACH-000502,PT-HGOuqa,ACH-000502,MC-000502-yIwz,ACH-000502,PT-HGOuqa,MC-000502-yIwz,STR-LoZQa3,False,"15, 16, 17","6, 7","28, 30, 30.2","16, 17, 18","7, 15, 19","8, 9, 11, 13","8, 12, 14","8, 11, 12","9, 11, 13","11, 12, 13","9, 10","17, 19","11, 12, 13, 16","11, 12","20, 23",NaN,0.6441,NaN,CX4010579,NaN,X,MC-000502-yIwz,Not Detected,NaN,PT-HGOuqa,Internal,GPP,NaN,TCCPAN2-encas12a,2024-08-13,2024-08-13 14:32:29.756766+00:00
889,ACH-000502,PT-HGOuqa,ACH-000502,MC-000502-yIwz,ACH-000502,PT-HGOuqa,MC-000502-yIwz,STR-kMG3Ry,False,"15, 17","7, 9.3","28, 30.2","17, 18","7, 15","8, 9","12, 14","11, 12","9, 13, 14","11, 12, 13","9, 10","17, 19","11, 12","11, 12",23,NaN,0.2500,NaN,CX4010882,NaN,X,MC-000502-yIwz,Not Detected,NaN,PT-HGOuqa,Internal,GPP,NaN,TCCPAN2-enCas12a-SetD-p4,2024-08-13,2024-08-13 14:32:29.756766+00:00
891,ACH-000502,PT-HGOuqa,ACH-000502,MC-000502-yIwz,ACH-000502,PT-HGOuqa,MC-000502-yIwz,STR-PeaGRb,False,"15, 17","7, 9.3","28, 30.2","17, 18","7, 15","8, 9","12, 14","11, 12","9, 13, 14","11, 12, 13","9, 10","17, 19","11, 12","11, 12",23,NaN,0.2500,NaN,CX4010790,NaN,X,MC-000502-yIwz,Not Detected,NaN,PT-HGOuqa,Internal,GPP,NaN,TCCPAN2-enCas12a-SetDp4,2024-08-13,2024-08-13 14:32:29.756766+00:00
2096,ACH-002459,PT-czDcsE,ACH-002459,MC-002459-cRI0,ACH-002459,PT-czDcsE,MC-002459-cRI0,STR-ywDfHZ,False,"14, 15","6, 9",NaN,15,12.1,"11, 14","11, 12",11,"12, 13",12,"8, 10",NaN,"13, 15",NaN,"23, 26",NaN,0.7442,NaN,NaN,NaN,NaN,MC-002459-cRI0,NaN,NaN,PT-czDcsE,Internal,WGS Inferred,NaN,NaN,2024-06-03,2024-06-03 16:39:03.919363+00:00
2111,ACH-001973,PT-oJTeSY,ACH-001973,MC-001973-ILv6,ACH-001973,PT-oJTeSY,MC-001973-ILv6,STR-FrA73B,False,14,NaN,32.3,"13, 17","10.1, 15.1","11, 12","11, 13",12,11,"11, 12","9, 11",NaN,"11, 13",NaN,"21, 23",NaN,0.7317,NaN,NaN,NaN,NaN,MC-001973-ILv6,NaN,NaN,PT-oJTeSY,Internal,WGS Inferred,NaN,NaN,2024-06-13,2024-06-13 16:41:08.205941+00:00
2124,ACH-001401,PT-7a0mgm,ACH-001401,MC-001401-l6Fd,ACH-001401,PT-7a0mgm,MC-001401-l6Fd,STR-o1zkEG,False,16,"7, 8",NaN,17,"7.1, 15.1",12,9,"8, 10",12,"11, 12","9, 12",NaN,10,NaN,20,NaN,0.7222,NaN,NaN,NaN,NaN,MC-001401-l6Fd,NaN,NaN,PT-7a0mgm,Internal,WGS Inferred,NaN,NaN,2024-04-10,2024-04-10 18:01:15.680098+00:00
2142,ACH-000093,PT-X8PfQy,ACH-000093,MC-000093-IB96,ACH-000093,PT-X8PfQy,MC-000093-IB96,STR-Lw91ng,False,"14, 16","6, 9.3",NaN,14,NaN,"12, 13","10, 12","9, 12","9, 11","12, 13","6, 8",NaN,"11, 13",NaN,"19, 23",NaN,0.1778,NaN,NaN,NaN,NaN,MC-000093-IB96,NaN,NaN,PT-X8PfQy,Internal,WGS Inferred,NaN,NaN,2024-06-03,2024-06-03 16:42:46.516476+00:00
2209,ACH-000201,PT-EY60tR,ACH-000201,MC-000201-rUIL,ACH-000201,PT-EY60tR,MC-000201-rUIL,STR-0gVv3Y,False,"17, 14","9.3, 8",31.3,NaN,"7.1, 10.1",12,"11, 12","10, 8","11, 12","10, 13","8, 12",NaN,"13, 15","9, 8",N

In [31]:
#STRs with percentage match >=0.75
df_above75 = all_str_single_upload[all_str_single_upload['percentage_match_to_parental'] >= 0.75]
df_above75.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1341 entries, 467 to 6483
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   model_id                      1280 non-null   object 
 1   model_patient_id              1280 non-null   object 
 2   model_model_id                1280 non-null   object 
 3   mc_model_condition_id         1280 non-null   object 
 4   mc_model_id                   1280 non-null   object 
 5   str_patient_id                1341 non-null   object 
 6   model_condition_id            1280 non-null   object 
 7   id                            1341 non-null   object 
 8   is_reference                  1341 non-null   bool   
 9   d3s1358                       1334 non-null   object 
 10  th01                          1329 non-null   object 
 11  d21s11                        817 non-null    object 
 12  d18s51                        1286 non-null   object 
 13  penta_

In [32]:
df_above75

,model_id,model_patient_id,model_model_id,mc_model_condition_id,mc_model_id,str_patient_id,model_condition_id,id,is_reference,d3s1358,th01,d21s11,d18s51,penta_e,d5s818,d13s317,d7s820,d16s539,csf1po,penta_d,vwa,d8s1179,tpox,fga,pellet_creation_date,percentage_match_to_parental,lab_corp_spec_nbr,lab_corp_case_nbr,comments,amelogenin,model_condition_id.1,mouse,mycoplasma,patient_id,source,source_group,pellet_submitted_date,sample_reference,created_at,updated_at
467,ACH-000970,PT-B2n6IN,ACH-000970,MC-000970-Bx3a,ACH-000970,PT-B2n6IN,MC-000970-Bx3a,STR-YuuzN8,False,15,"6, 9","29, 31.2","13, 15, 19, 20, 21, 22","19, 20","11, 12","8, 9",12,"8, 12, 13, 14","11, 13","10, 12, 13",15,"12, 14, 15",8,"19, 22, 23",2016-09-27,0.8308,NaN,CX4014974,NaN,X,MC-000970-Bx3a,NaN,NaN,PT-B2n6IN,Internal,GPP,NaN,SNU-C5-311cas9(9/27/2016),2024-08-13,2024-08-13 14:32:29.468693+00:00
469,ACH-000970,PT-B2n6IN,ACH-000970,MC-000970-Bx3a,ACH-000970,PT-B2n6IN,MC-000970-Bx3a,STR-OIfYbO,False,15,"6, 9","29, 31.2","15, 20, 21, 22",20,"11, 12",8,12,"8, 12, 13, 14","10, 11, 13","12, 13",15,"12, 14, 15",8,"18, 19, 22, 23",2023-07-07,0.8710,NaN,CX4016892,NaN,X,MC-000970-Bx3a,NaN,NaN,PT-B2n6IN,Internal,GPP,NaN,SNUC5-Cas9VY,2024-08-13,2024-08-13 14:32:29.468693+00:00
470,ACH-000970,PT-B2n6IN,ACH-000970,MC-000970-Bx3a,ACH-000970,PT-B2n6IN,MC-000970-Bx3a,STR-xW7cbo,False,15,"6, 9","29, 31.2","15, 20, 21, 22","20, 21","11, 12, 13",8,12,"8, 13","11, 12, 13, 14","12, 13",15,"12, 14, 15",8,"18, 19, 22, 23",2023-11-09,0.8254,NaN,CX4017930,NaN,X,MC-000970-Bx3a,NaN,NaN,PT-B2n6IN,Internal,GPP,NaN,SNUC5-Cas9VY-ParalogV2-RepA-P6(D22),2024-08-13,2024-08-13 14:32:29.468693+00:00
471,ACH-000970,PT-B2n6IN,ACH-000970,MC-000970-Bx3a,ACH-000970,PT-B2n6IN,MC-000970-Bx3a,STR-eFqrqg,False,"15, 16, 17","6, 9","29, 31.2","15, 20, 21, 22","20, 21","11, 12",8,12,"8, 13, 14, 15","11, 13, 14","12, 13",15,"12, 14, 15",8,"18, 19, 23",2023-10-19,0.8125,NaN,CX4017728,NaN,X,MC-000970-Bx3a,NaN,NaN,PT-B2n6IN,Internal,GPP,NaN,SNUC5-Cas9VY-ParalogV2-RepB-P6(D22),2024-08-13,2024-08-13 14:32:29.468693+00:00
481,ACH-003099,PT-BHjOZY,ACH-003099,MC-003099-DM4v,ACH-003099,PT-BHjOZY,MC-003099-DM4v,STR-5Vx5yk,False,16,"6, 7",30,15,"12, 13",12,"8, 12","10, 12","9, 13","11, 12","8, 9","16, 18","10, 13","8, 11","20, 25",2022-10-21,1.0000,NaN,CX4014974,NaN,X,MC-003099-DM4v,NaN,NaN,PT-BHjOZY,Internal,GPP,NaN,UCSFOT1109-enCas12a,2024-08-13,2024-08-13 14:32:29.468693+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6466,ACH-001031,PT-nuzviS,ACH-001031,MC-001031-oRDI,ACH-001031,PT-nuzviS,MC-001031-oRDI,STR-E42whd,False,"15, 18","6, 7",31.3,"17, 19",NaN,"10, 12","10, 9","10, 12","10, 12","10, 11","12, 13",NaN,"10, 14",NaN,"24, 23",NaN,0.8571,NaN,NaN,NaN,NaN,MC-001031-oRDI,NaN,NaN,PT-nuzviS,Internal,WGS Inferred,NaN,NaN,2024-04-09,2024-04-09 16:00:46.668811+00:00
6467,ACH-000449,PT-E7giyP,ACH-000449,MC-000449-iHb5,ACH-000449,PT-E7giyP,MC-000449-iHb5,STR-E6KW1y,False,"14, 17",6,NaN,"12, 15","7.1, 10.1",13,"11, 13","7, 11","11, 12",11,"12, 14",NaN,"14, 15",NaN,"20, 25",NaN,0.7556,NaN,NaN,NaN,NaN,MC-000449-iHb5,NaN,NaN,PT-E7giyP,Internal,WGS Inferred,NaN,NaN,2024-06-13,2024-06-13 16:48:18.244183+00:00
6477,ACH-002968,PT-pjhrsc,ACH-002968,MC-002968-WZFJ,ACH-002968,PT-pjhrsc,MC-002968-WZFJ,STR-JhhHTm,False,"15, 18",9.3,"30, 32.2",14,"5, 7","11, 12",12,11,12,"9, 12",10,17,"15, 16",9,"23, 24",NaN,1.0000,29G90970,CX4014631,NaN,X,MC-002968-WZFJ,Not Detected,NaN,PT-pjhrsc,Internal,Minerva,NaN,ACH-002968 Post H1 FP (pX+11),2024-04-09,2024-04-09 16:00:46.668811+00:00
6480,ACH-001310,PT-1UiNjR,ACH-001310,MC-001310-xz7e,ACH-001310,PT-1UiNjR,MC-001310-xz7e,STR-m7DSLR,False,"14, 17","6, 9.3",NaN,"12, 16","12.1, 15.1","11, 12",12,"9, 12","9, 13","10, 11","10, 13",NaN,"13, 15",NaN,"20, 22",NaN,0.7917,NaN,NaN,NaN,NaN,MC-001310-xz7e,NaN,NaN,PT-1UiNjR,Internal,WGS Inferred,NaN,NaN,2024-05-24,2024-05-24 16:43:47.000144+00:00


In [33]:
#reference
all_str_single_upload['is_reference'].value_counts()

is_reference
False    5394
True     1092
Name: count, dtype: int64

In [34]:
#source
all_str_single_upload['source'].value_counts()

source
Internal    6401
External      85
Name: count, dtype: int64

In [35]:
#source groups
all_str_single_upload['source_group'].value_counts()

source_group
WGS Inferred                2196
GPP                         2122
DMX                          783
PRISM                        718
Minerva                      452
Mnemosyne                    135
Achilles                      26
Cellosaurus                   25
Oslo University Hospital      23
CCLF                           1
Name: count, dtype: int64

In [36]:
#total counts of reference by patient id
counts = all_str_single_upload.groupby('str_patient_id')['is_reference'].value_counts()
counts = counts.reset_index(name='count')
counts.head(15)

,str_patient_id,is_reference,count
0,PT-00UDx5,False,3
1,PT-00UDx5,True,1
2,PT-01q6LP,False,2
3,PT-01q6LP,True,1
4,PT-03AcoW,False,5
5,PT-03AcoW,True,1
6,PT-04HLfu,True,1
7,PT-04HLfu,False,1
8,PT-04KbrE,False,1
9,PT-0531Uw,False,3


In [37]:
#combined
ref_counts = all_str_single_upload.groupby(['str_patient_id', 'is_reference']).size().unstack(fill_value=0)
ref_counts.to_csv('ref_counts.csv')
ref_counts.head(50)

is_reference,False,True
str_patient_id,,
PT-00UDx5,3,1
PT-01q6LP,2,1
PT-03AcoW,5,1
PT-04HLfu,1,1
PT-04KbrE,1,0
PT-0531Uw,3,1
PT-05dldv,2,0
PT-05hA76,1,1
PT-0Ds3by,1,0


In [38]:
#sample str no ref str
ref_counts.columns = [ 'False', 'True']
no_refs = ref_counts[(ref_counts['False'] >= 1) & (ref_counts['True'] == 0)]
no_refs.to_csv('str_with_no_refs.csv')
no_refs

,False,True
str_patient_id,,
PT-04KbrE,1,0
PT-05dldv,2,0
PT-0Ds3by,1,0
PT-0F4qCF,3,0
PT-0Fp6tq,4,0
...,...,...
PT-zlBs5P,3,0
PT-zp1phR,1,0
PT-zqsQMh,1,0


In [39]:
#ref str but no sample str
ref_no_sample = ref_counts[(ref_counts['False'] == 0) & (ref_counts['True'] >= 1)]
ref_no_sample.to_csv('str_with_refs_no_sample_strs.csv')
ref_no_sample

,False,True
str_patient_id,,
PT-0uXvN6,0,1
PT-0zEk9y,0,1
PT-1Ii4Q3,0,1
PT-1rbGNh,0,1
PT-2KZg5H,0,1
PT-2UMZn1,0,1
PT-2a1cpe,0,1
PT-2gvMHE,0,1
PT-2iz8vp,0,1


In [40]:
#sample str and ref str
both_ref_sample_str = ref_counts[(ref_counts['False'] >= 1) & (ref_counts['True'] >= 1)]
both_ref_sample_str.to_csv('str_with_both_refs.csv')
both_ref_sample_str

,False,True
str_patient_id,,
PT-00UDx5,3,1
PT-01q6LP,2,1
PT-03AcoW,5,1
PT-04HLfu,1,1
PT-0531Uw,3,1
...,...,...
PT-zfPfIU,6,1
PT-zjOA3C,1,1
PT-zuTmVR,3,1


In [41]:
#including model_conditions
ref_counts_mod_cond = all_str_single_upload.groupby(['str_patient_id','model_condition_id', 'is_reference']).size().unstack(fill_value=0)
ref_counts_mod_cond

is_reference                       False  True 
str_patient_id model_condition_id              
PT-00UDx5      MC-003044-gcIN          3      1
PT-01q6LP      MC-000000-TEST          1      0
               MC-001541-ux7W          0      1
PT-03AcoW      MC-002708-tPxj          5      1
PT-04HLfu      MC-001533-SowA          1      1
...                                  ...    ...
PT-zwxgMM      MC-001484-dooe          1      1
               MC-001485-Qw1Y          2      0
PT-zx0MVs      MC-001865-3Vel          0      1
PT-zy2Tek      MC-003230-Uu6g          1      1
PT-zzULgh      MC-001913-ceWs          0      1

[2170 rows x 2 columns]

In [42]:
#ref str but no sample str with by model condition
ref_counts_mod_cond.columns = ['False', 'True']
ref_only_mod_cond = ref_counts_mod_cond[(ref_counts_mod_cond['False'] == 0) & (ref_counts_mod_cond['True'] == 1)]
ref_only_mod_cond

,,False,True
str_patient_id,model_condition_id,,
PT-01q6LP,MC-001541-ux7W,0,1
PT-0uXvN6,MC-001678-DptN,0,1
PT-0zEk9y,MC-001881-Ffwd,0,1
PT-1Ii4Q3,MC-001884-KhOL,0,1
PT-1rbGNh,MC-001907-Z0iF,0,1
PT-2KZg5H,MC-001361-ghvT,0,1
PT-2UMZn1,MC-001891-KFqz,0,1
PT-2a1cpe,MC-001620-6V1R,0,1
PT-2gvMHE,MC-001493-yY6r,0,1


In [43]:
#sample str no ref str by model condition
no_refs_mod_cond = ref_counts_mod_cond[(ref_counts_mod_cond['False'] >= 1) & (ref_counts_mod_cond['True'] == 0)]
no_refs_mod_cond

,,False,True
str_patient_id,model_condition_id,,
PT-01q6LP,MC-000000-TEST,1,0
PT-04KbrE,MC-000722-Bobx,1,0
PT-05dldv,MC-000979-bkVz,1,0
PT-0Ds3by,MC-000297-kYTw,1,0
PT-0E9rgp,MC-002822-EnEF,2,0
...,...,...,...
PT-zp1phR,MC-003629-S2H9,1,0
PT-zqsQMh,MC-003105-cFPR,1,0
PT-zrYGap,MC-000914-GsyT,1,0


In [44]:
#both sample and ref str with model condition
no_refs_mod_cond_both = ref_counts_mod_cond[(ref_counts_mod_cond['False'] >= 1) & (ref_counts_mod_cond['True'] >= 1)]
no_refs_mod_cond_both

,,False,True
str_patient_id,model_condition_id,,
PT-00UDx5,MC-003044-gcIN,3,1
PT-03AcoW,MC-002708-tPxj,5,1
PT-04HLfu,MC-001533-SowA,1,1
PT-0531Uw,MC-002052-GfvX,3,1
PT-05hA76,MC-001680-sp3j,1,1
...,...,...,...
PT-zfPfIU,MC-002964-0orR,4,1
PT-zjOA3C,MC-001402-CPYn,1,1
PT-zuTmVR,MC-003334-3g7u,3,1
